# Question Answering

In [ ]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

persist_directory = "../data/chroma/"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
print(vectordb._collection.count())

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

### RetrievalQA chain — default

In [ ]:
from langchain_classic.chains import RetrievalQA

question = "هل يجوز لصاحب العمل فصل عامل تغيب عن العمل دون انذار؟"

qa_chain = RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever(search_kwargs={"k": 2}))
result = qa_chain.invoke({"query": question})
result["result"]

### Prompt — Arabic, with strict citation rules

In [ ]:
from langchain_core.prompts import PromptTemplate

SYSTEM_TEMPLATE = """انت مساعد قانوني متخصص في القانون البحريني. استخدم المقاطع القانونية التالية فقط للاجابة على السؤال في نهاية النص.

قواعد صارمة يجب اتباعها:
- استند فقط الى النصوص المرفقة، ولا تخترع اي معلومة غير موجودة فيها.
- اذا لم تكن الاجابة موجودة في النصوص المرفقة، صرح بذلك بوضوح ولا تخمن.
- اذكر المصدر الدقيق لكل معلومة (رقم المادة او رقم القضية).
- اذا استندت الاجابة الى اكثر من قانون او حكم، اذكرهم جميعا.

النصوص القانونية:
{context}

السؤال: {question}

الاجابة القانونية المدعومة بالمصادر:"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(SYSTEM_TEMPLATE)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
)

result = qa_chain.invoke({"query": question})
print(result["result"])

In [ ]:
for doc in result["source_documents"]:
    print(doc.metadata)

### Chain types — `stuff` vs `map_reduce` vs `refine`

In [ ]:
qa_chain_map_reduce = RetrievalQA.from_chain_type(
    llm, retriever=vectordb.as_retriever(search_kwargs={"k": 2}), chain_type="map_reduce"
)
qa_chain_refine = RetrievalQA.from_chain_type(
    llm, retriever=vectordb.as_retriever(search_kwargs={"k": 2}), chain_type="refine"
)

print("--- map_reduce ---")
print(qa_chain_map_reduce.invoke({"query": question})["result"])
print("\n--- refine ---")
print(qa_chain_refine.invoke({"query": question})["result"])

### RetrievalQA limitations

In [ ]:
follow_up = "لماذا يشترط القانون الانذار قبل الفصل؟"
result = qa_chain.invoke({"query": follow_up})
result["result"]